In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm
from pathlib import Path
import numpy as np
import re

def parse_file_info(path):
    name = Path(path).stem
    
    location = name.split("_")[0]
    
    if re.search(r"precip|rain", name, re.IGNORECASE):
        variable = "precipitation"
    elif re.search(r"temp", name, re.IGNORECASE):
        variable = "temperature"
    else:
        variable = "unknown"
        
    return location, variable

In [2]:
import pandas as pd
import numpy as np

def read_climate_excel(path):
    # Read
    df = pd.read_excel(path)
    
    # Rename first col → year
    df.rename(columns={df.columns[0]: "year"}, inplace=True)
    
    # Drop column 'Annual' if present
    df = df.drop(columns=[c for c in df.columns if c.lower() == "annual"], errors="ignore")
    
    # Standardize month names (fix typos)
    month_fix = {
        "Octomber": "October",
        "Octomber ": "October",
        "Septembar": "September"
    }
    df.rename(columns=month_fix, inplace=True)
    
    # Convert all month columns to numeric
    for col in df.columns:
        if col != "year":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Melt to long format (tidy)
    df_long = df.melt(
        id_vars="year",
        var_name="month",
        value_name="value"
    )
    
    # Remove missing months (e.g. broken cells)
    df_long = df_long.dropna(subset=["month"])
    
    # Fix capitalization
    df_long["month"] = df_long["month"].str.title()
    
    # Month → number
    month_map = {m:i for i,m in enumerate([
        "", "January","February","March","April","May","June",
        "July","August","September","October","November","December"
    ])}
    
    df_long["month_num"] = df_long["month"].map(month_map)
    
    # Build date
    df_long["date"] = pd.to_datetime(
        dict(year=df_long["year"], month=df_long["month_num"], day=1),
        errors="coerce"
    )
    
    return df_long

In [5]:
from glob import glob
from tqdm import tqdm

data_dir = "datasets/*.xlsx"
files = glob(data_dir)

all_data = []

for f in tqdm(files):
    loc, var = parse_file_info(f)
    df = read_climate_excel(f)
    df["location"] = loc
    df["variable"] = var
    all_data.append(df)

climate_all = pd.concat(all_data, ignore_index=True)


100%|██████████| 24/24 [10:23<00:00, 25.99s/it]


In [6]:
climate_all

,year,month,value,month_num,date,location,variable
0,1870.0,January,76.0,1.0,1870-01-01,Anuradhapura,precipitation
1,1871.0,January,287.0,1.0,1871-01-01,Anuradhapura,precipitation
2,1872.0,January,38.0,1.0,1872-01-01,Anuradhapura,precipitation
3,1873.0,January,5.0,1.0,1873-01-01,Anuradhapura,precipitation
4,1874.0,January,33.0,1.0,1874-01-01,Anuradhapura,precipitation
...,...,...,...,...,...,...,...
152065630,NaN,Unnamed: 13,NaN,NaN,NaT,Trincomalee,temperature
152065631,NaN,Unnamed: 13,NaN,NaN,NaT,Trincomalee,temperature
152065632,NaN,Unnamed: 13,NaN,NaN,NaT,Trincomalee,temperature
152065633,NaN,Unnamed: 13,NaN,NaN,NaT,Trincomalee,temperature


In [17]:
climate_all['month'].unique()

array(['January', 'February', 'March', 'April', 'May', 'June', 'July',
       'August', 'September', 'Octomber', 'November', 'December',
       'October', 'Desember', 'Annual'], dtype=object)

In [18]:
climate_all

,year,month,value,month_num,date,location,variable
0,1870.0,January,76.0,1.0,1870-01-01,Anuradhapura,precipitation
1,1871.0,January,287.0,1.0,1871-01-01,Anuradhapura,precipitation
2,1872.0,January,38.0,1.0,1872-01-01,Anuradhapura,precipitation
3,1873.0,January,5.0,1.0,1873-01-01,Anuradhapura,precipitation
4,1874.0,January,33.0,1.0,1874-01-01,Anuradhapura,precipitation
...,...,...,...,...,...,...,...
152065630,NaN,Annual,NaN,NaN,NaT,Trincomalee,temperature
152065631,NaN,Annual,NaN,NaN,NaT,Trincomalee,temperature
152065632,NaN,Annual,NaN,NaN,NaT,Trincomalee,temperature
152065633,NaN,Annual,NaN,NaN,NaT,Trincomalee,temperature


In [19]:
climate_all.to_csv("climate_data_combined.csv", index=False)